# {mod}`tkinter` 异步任务

{mod}`asyncio` 专门实现 Concurrency and Multithreading（多线程和并发）的函数:
1. `loop.call_soon_threadsafe(callback, *args)`: 很底层的 API 接口，一般很少使用，本文也暂时不做讨论。
2. `asyncio.run_coroutine_threadsafe(coroutine，loop)`

`asyncio.run_coroutine_threadsafe(coroutine，loop)` 第一个参数为需要异步执行的协程函数，第二个 `loop` 参数为在新线程中创建的事件循环 loop，注意一定要是在新线程中创建哦，该函数的返回值是一个concurrent.futures.Future类的对象，用来获取协程的返回结果。
```python
future = asyncio.run_coroutine_threadsafe(coro_func(), loop)   # 在新线程中运行协程
result = future.result()   #等待获取Future的结果
```

In [ ]:
import asyncio 
 
import threading
 
#需要执行的耗时异步任务
async def func(num):
    print(f'准备调用func, 大约耗时{num}')
    await asyncio.sleep(num)
    print(f'耗时{num}之后, func函数运行结束')
 
#定义一个专门创建事件循环loop的函数，在另一个线程中启动它
def start_loop(loop):
    asyncio.set_event_loop(loop)
    loop.run_forever()
 
#定义一个main函数
def main():
    coroutine1 = func(3)
    coroutine2 = func(2)
    coroutine3 = func(1)
 
    new_loop = asyncio.new_event_loop()                        #在当前线程下创建时间循环，（未启用），在start_loop里面启动它
    t = threading.Thread(target=start_loop, args=(new_loop,))   #通过当前线程开启新的线程去启动事件循环
    t.start()
 
    asyncio.run_coroutine_threadsafe(coroutine1, new_loop)  #这几个是关键，代表在新线程中事件循环不断“游走”执行
    asyncio.run_coroutine_threadsafe(coroutine2, new_loop)
    asyncio.run_coroutine_threadsafe(coroutine3, new_loop)
 
    for i in "iloveu":
        print(str(i)+"    ")
 
if __name__ == "__main__":
    main()

准备调用func, 大约耗时3
准备调用func, 大约耗时2
i    
l    
o    
v    
e    
u    
准备调用func, 大约耗时1


耗时1之后, func函数运行结束
耗时2之后, func函数运行结束
耗时3之后, func函数运行结束


`main` 是在主线程中的，而三个协程函数是在新线程中的，它们是在一起执行的，没有造成主线程 `main` 的阻塞。

## `tkinter`+`threading`+`asyncio`

```python
from tkinter import Tk, ttk
import asyncio
import threading

class Window(Tk):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.geometry('500x300')
        self.button = ttk.Button(self, text="开始计算", command=self.change_form_state)
        self.label = ttk.Label(master=self, text="等待计算结果")
        self.button.pack()
        self.label.pack()

    async def calculate(self):
        await asyncio.sleep(3)
        self.label["text"] = 300
 
    def get_loop(self,loop):
        self.loop = loop
        asyncio.set_event_loop(self.loop)
        self.loop.run_forever()
    
    def change_form_state(self):
        self.label["text"] = "等待计算结果"
        coroutine1 = self.calculate()
        new_loop = asyncio.new_event_loop()                        #在当前线程下创建时间循环，（未启用），在start_loop里面启动它
        t = threading.Thread(target=self.get_loop,args=(new_loop,))   #通过当前线程开启新的线程去启动事件循环
        t.start()
        asyncio.run_coroutine_threadsafe(coroutine1, new_loop)  #这几个是关键，代表在新线程中事件循环不断“游走”执行

root = Window()
root.mainloop()
```